In [1]:
import os
import glob
import json
import time
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [2]:
def env_int(name, default):
    v = os.getenv(name)
    return default if v is None else int(v)

def env_float(name, default):
    v = os.getenv(name)
    return default if v is None else float(v)

def env_str(name, default):
    v = os.getenv(name)
    return default if v is None else str(v)

SEED = env_int("SEED", 42)

N_CITIES   = env_int("N_CITIES", 210)
EPOCHS     = env_int("EPOCHS", 50)
BATCH_SIZE = env_int("BATCH_SIZE", 8192)

WIDTH   = env_int("WIDTH", 1024)
DEPTH   = env_int("DEPTH", 8)
DROPOUT = env_float("DROPOUT", 0.1)

LAGS  = tuple(int(x) for x in env_str("LAGS", "1,3,7,14,30,60").split(","))
ROLLS = tuple(int(x) for x in env_str("ROLLS", "7,30").split(","))

NUM_WORKERS = env_int("NUM_WORKERS", 8)
PREFETCH    = env_int("PREFETCH_FACTOR", 4)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (DEVICE.type == "cuda")
PIN_MEMORY = (DEVICE.type == "cuda")

print("DEVICE:", DEVICE)
print("USE_AMP:", USE_AMP)
print("N_CITIES/EPOCHS/BATCH_SIZE:", N_CITIES, EPOCHS, BATCH_SIZE)
print("WIDTH/DEPTH/DROPOUT:", WIDTH, DEPTH, DROPOUT)
print("LAGS/ROLLS:", LAGS, ROLLS)
print("NUM_WORKERS:", NUM_WORKERS, "PIN_MEMORY:", PIN_MEMORY)

os.makedirs("results/benchmarks", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)
os.makedirs("outputs/models", exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

DEVICE: cuda
USE_AMP: True
N_CITIES/EPOCHS/BATCH_SIZE: 210 50 8192
WIDTH/DEPTH/DROPOUT: 1024 8 0.1
LAGS/ROLLS: (1, 3, 7, 14, 30, 60) (7, 30)
NUM_WORKERS: 8 PIN_MEMORY: True


### Pointing towards the dataset

In [3]:
DATA_DIR = "7890488"
CITY_INFO_PATH = os.path.join(DATA_DIR, "city_info.csv")

assert os.path.exists(DATA_DIR), f"Missing folder: {DATA_DIR}"
assert os.path.exists(CITY_INFO_PATH), f"Missing file: {CITY_INFO_PATH}"

city_info = pd.read_csv(CITY_INFO_PATH)
city_info.head(), city_info.shape

(   Unnamed: 0      Name           ID      Lat       Lon  \
 0           1    Lander  USW00024021  42.8153 -108.7261   
 1           2    Lander  USW00024021  42.8153 -108.7261   
 2           3  Cheyenne  USW00024018  41.1519 -104.8061   
 3           4  Cheyenne  USW00024018  41.1519 -104.8061   
 4           5    Wausau  USW00014897  44.9258  -89.6256   
 
                   Stn.Name  Stn.stDate  Stn.edDate  
 0               LANDER WBO  1892-01-01  1946-05-28  
 1        LANDER HUNT FIELD  1946-05-29  2023-12-31  
 2             CHEYENNE WBO  1871-01-01  1935-08-31  
 3  CHEYENNE MUNICIPAL ARPT  1935-09-01  2023-12-31  
 4     Wausau Record Herald  1896-01-01  1941-12-31  ,
 (461, 8))

### Identify individual city files

In [4]:
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
city_files = [p for p in all_files if os.path.basename(p).lower() != "city_info.csv"]

print("Total CSV files:", len(all_files))
print("City files:", len(city_files))
print("Example:", os.path.basename(city_files[0]) if city_files else None)

Total CSV files: 211
City files: 210
Example: USW00094014.csv


### Load one city at a time 

In [5]:
sample_path = city_files[0]
df0 = pd.read_csv(sample_path)
df0.head(), df0.columns, df0.shape

(   Unnamed: 0        Date  tmax  tmin  prcp
 0           1  1894-01-01  32.0  10.0   0.0
 1           2  1894-01-02  -6.0 -12.0   0.0
 2           3  1894-01-03   0.0 -20.0   0.0
 3           4  1894-01-04   6.0 -14.0   0.0
 4           5  1894-01-05  -7.0 -18.0   0.0,
 Index(['Unnamed: 0', 'Date', 'tmax', 'tmin', 'prcp'], dtype='object'),
 (47481, 5))

### Handling common naming variations

In [6]:
def standardize_city_df(df: pd.DataFrame, city_id: str) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}

    date_col = cols.get("date")
    tmax_col = cols.get("tmax") 
    tmin_col = cols.get("tmin")
    prcp_col = cols.get("prcp")

    out = df[[date_col, tmax_col, tmin_col, prcp_col]].copy()
    out.columns = ["date", "tmax", "tmin", "prcp"]
    out["date"] = pd.to_datetime(out["date"])
    out["city_id"] = city_id

    for c in ["tmax", "tmin", "prcp"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.sort_values("date").reset_index(drop=True)
    return out

### Loading `n` cities

In [7]:
def city_id_from_path(p: str) -> str:
    return os.path.splitext(os.path.basename(p))[0]

N_CITIES = 25
selected_paths = city_files[:N_CITIES]

frames = []
for p in tqdm(selected_paths):
    cid = city_id_from_path(p)
    df = pd.read_csv(p)
    frames.append(standardize_city_df(df, cid))

data = pd.concat(frames, ignore_index=True)
data.head(), data.shape

  0%|          | 0/25 [00:00<?, ?it/s]

(        date  tmax  tmin  prcp      city_id
 0 1894-01-01  32.0  10.0   0.0  USW00094014
 1 1894-01-02  -6.0 -12.0   0.0  USW00094014
 2 1894-01-03   0.0 -20.0   0.0  USW00094014
 3 1894-01-04   6.0 -14.0   0.0  USW00094014
 4 1894-01-05  -7.0 -18.0   0.0  USW00094014,
 (1266651, 5))

In [8]:
lower_cols = {c.lower(): c for c in city_info.columns}
print("city_info columns:", city_info.columns.tolist())

CITY_ID_COL = lower_cols.get("id")
LAT_COL = lower_cols.get("lat")
LON_COL = lower_cols.get("lon")

assert CITY_ID_COL and LAT_COL and LON_COL, "Fix CITY_ID_COL/LAT_COL/LON_COL based on city_info columns."

meta = city_info[[CITY_ID_COL, LAT_COL, LON_COL]].copy()
meta.columns = ["city_id", "lat", "lon"]

data = data.merge(meta, on="city_id", how="left")
data[["lat","lon"]].isna().mean()

city_info columns: ['Unnamed: 0', 'Name', 'ID', 'Lat', 'Lon', 'Stn.Name', 'Stn.stDate', 'Stn.edDate']


lat    0.0
lon    0.0
dtype: float64

### Feature engineering

In [9]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["dayofyear"] = d["date"].dt.dayofyear
    d["sin_doy"] = np.sin(2*np.pi*d["dayofyear"]/365.25)
    d["cos_doy"] = np.cos(2*np.pi*d["dayofyear"]/365.25)
    return d

def add_lags_rolls(df: pd.DataFrame, lags=LAGS, rolls=ROLLS) -> pd.DataFrame:
    d = df.copy().sort_values(["city_id", "date"])
    g = d.groupby("city_id", sort=False)

    base_cols = ["tmax", "tmin", "prcp"]

    for lag in lags:
        for col in base_cols:
            d[f"{col}_lag{lag}"] = g[col].shift(lag)

    for col in base_cols:
        d[f"{col}_diff1"] = g[col].diff(1)

    for win in rolls:
        for col in base_cols:
            r = g[col].shift(1).rolling(win)
            d[f"{col}_roll{win}_mean"] = r.mean()
            d[f"{col}_roll{win}_std"]  = r.std()

    return d

data_fe = add_time_features(data)
data_fe = add_lags_rolls(data_fe)

for col in ["tmax", "tmin", "prcp"]:
    data_fe[f"{col}_target"] = data_fe.groupby("city_id")[col].shift(-1)

feature_cols = (
    ["lat", "lon", "sin_doy", "cos_doy"]
    + [f"{c}_lag{l}" for l in LAGS for c in ("tmax","tmin","prcp")]
    + [f"{c}_diff1" for c in ("tmax","tmin","prcp")]
    + [f"{c}_roll{w}_mean" for w in ROLLS for c in ("tmax","tmin","prcp")]
    + [f"{c}_roll{w}_std"  for w in ROLLS for c in ("tmax","tmin","prcp")]
)
target_cols = ["tmax_target", "tmin_target", "prcp_target"]

model_df = data_fe.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
print("model_df shape:", model_df.shape)
print("n_features:", len(feature_cols))

model_df shape: (2430377, 46)
n_features: 37


### Time based split

In [10]:
dates = model_df["date"]
train_end = pd.Timestamp("2016-12-31")
val_end   = pd.Timestamp("2019-12-31")

train_df = model_df[dates <= train_end].copy()
val_df   = model_df[(dates > train_end) & (dates <= val_end)].copy()
test_df  = model_df[dates > val_end].copy()

len(train_df), len(val_df), len(test_df), (train_df["date"].min(), test_df["date"].max())

(2292375,
 60271,
 77731,
 (Timestamp('1872-11-19 00:00:00'), Timestamp('2023-12-31 00:00:00')))

### Scaling features

In [11]:
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df[target_cols].values.astype(np.float32)
train_city_ids = train_df["city_id"].to_numpy()

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df[target_cols].values.astype(np.float32)
val_city_ids = val_df["city_id"].to_numpy()

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df[target_cols].values.astype(np.float32)
test_city_ids = test_df["city_id"].to_numpy()


scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train).astype(np.float32)
X_val_s   = scaler_X.transform(X_val).astype(np.float32)
X_test_s  = scaler_X.transform(X_test).astype(np.float32)

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(
    TabDataset(X_train_s, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
)

val_loader = DataLoader(
    TabDataset(X_val_s, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
)

test_loader = DataLoader(
    TabDataset(X_test_s, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
)

### GPU model

In [12]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim=3, width=WIDTH, depth=DEPTH, dropout=DROPOUT):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [
                nn.Linear(d, width),
                nn.LayerNorm(width),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            d = width
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP(in_dim=len(feature_cols), out_dim=len(target_cols)).to(DEVICE)
print("Model params:", sum(p.numel() for p in model.parameters()))

Model params: 7405571


In [13]:
loss_fn = nn.SmoothL1Loss(beta=1.0)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

scaler_amp = torch.cuda.amp.GradScaler(enabled=USE_AMP)


/tmp/ipykernel_2590500/1437620048.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler(enabled=USE_AMP)


### Training loop

In [14]:
def run_epoch(model, loader, train: bool):
    model.train(train)
    total_loss = 0.0
    n = 0

    for Xb, yb in loader:
        Xb = Xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        if train:
            opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = model(Xb)
            loss = loss_fn(pred, yb)

        if train:
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt)
            scaler_amp.update()

        bs = Xb.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / max(n, 1)

history = []
start = time.time()

train_samples = len(train_loader.dataset)
epoch_rows = []

total_train_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()

    tr_loss = run_epoch(model, train_loader, train=True)
    va_loss = run_epoch(model, val_loader, train=False)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    epoch_end = time.perf_counter()
    epoch_sec = epoch_end - epoch_start
    throughput = train_samples / epoch_sec if epoch_sec > 0 else None

    row = {
        "epoch": epoch,
        "train_loss": tr_loss,
        "val_loss": va_loss,
        "epoch_sec": epoch_sec,
        "throughput_samples_per_sec": throughput,
    }
    epoch_rows.append(row)

    print(
        f"Epoch {epoch:02d} | train {tr_loss:.4f} | val {va_loss:.4f} | "
        f"time {epoch_sec:.2f}s | throughput {throughput:.1f} samples/s"
    )

total_train_end = time.perf_counter()
total_train_sec = total_train_end - total_train_start

/tmp/ipykernel_2590500/1999259727.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 01 | train 17.9262 | val 7.5168 | time 4.19s | throughput 547334.7 samples/s


Epoch 02 | train 4.0320 | val 2.8664 | time 4.37s | throughput 525166.0 samples/s


Epoch 03 | train 2.8439 | val 2.4813 | time 4.95s | throughput 462812.1 samples/s


Epoch 04 | train 2.0716 | val 1.6567 | time 5.56s | throughput 412189.2 samples/s


...

Epoch 46 | train 1.4873 | val 1.2960 | time 5.42s | throughput 422708.0 samples/s


Epoch 47 | train 1.4851 | val 1.2958 | time 4.97s | throughput 460842.7 samples/s


Epoch 48 | train 1.4809 | val 1.3077 | time 4.50s | throughput 508997.6 samples/s


Epoch 49 | train 1.4787 | val 1.2897 | time 4.79s | throughput 478677.1 samples/s


Epoch 50 | train 1.4774 | val 1.2967 | time 4.92s | throughput 466175.8 samples/s


In [15]:
epoch_df = pd.DataFrame(epoch_rows)
epoch_df.to_csv("outputs/metrics/epoch_timing.csv", index=False)

summary = {
    "total_training_sec": total_train_sec,
    "total_training_min": total_train_sec / 60.0,
    "avg_epoch_sec": float(epoch_df["epoch_sec"].mean()),
    "avg_throughput_samples_per_sec": float(epoch_df["throughput_samples_per_sec"].mean()),
}
with open("outputs/metrics/run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary

{'total_training_sec': 239.18204026902094,
 'total_training_min': 3.9863673378170157,
 'avg_epoch_sec': 4.783355395840481,
 'avg_throughput_samples_per_sec': 481975.7304578958}

In [16]:
import os, json, platform
from datetime import datetime

PLATFORM_LABEL = os.getenv("PLATFORM_LABEL", "unknown-platform")

RUN_META = {
    "run_id": datetime.utcnow().strftime("%Y%m%d_%H%M%SZ"),
    "platform_label": PLATFORM_LABEL,
    "instance_label": platform.node(),
    "cpu_or_gpu": "GPU" if DEVICE.type == "cuda" else "CPU",
    "torch_cuda_available": bool(torch.cuda.is_available()),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "device": str(DEVICE),
    "seed": SEED,
    "n_cities": N_CITIES,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "n_features": len(feature_cols),
    "model_params": int(sum(p.numel() for p in model.parameters())),
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "prefetch_factor": PREFETCH,
    "amp_enabled": USE_AMP,
    "width": WIDTH, "depth": DEPTH, "dropout": DROPOUT,
    "lags": list(LAGS), "rolls": list(ROLLS),
    "n_train_rows": int(len(train_df)),
    "n_val_rows": int(len(val_df)),
    "n_test_rows": int(len(test_df)),
}
if DEVICE.type == "cuda":
    RUN_META["gpu_name"] = torch.cuda.get_device_name(0)

with open("results/benchmarks/run_metadata.json", "w") as f:
    json.dump(RUN_META, f, indent=2)

RUN_META

{'run_id': '20260303_005030Z',
 'platform_label': 'Anvil',
 'instance_label': 'h004.anvil.rcac.purdue.edu',
 'cpu_or_gpu': 'GPU',
 'torch_cuda_available': True,
 'cuda_visible_devices': '0',
 'device': 'cuda',
 'seed': 42,
 'n_cities': 25,
 'batch_size': 8192,
 'epochs': 50,
 'n_features': 37,
 'model_params': 7405571,
 'num_workers': 8,
 'pin_memory': True,
 'prefetch_factor': 4,
 'amp_enabled': True,
 'width': 1024,
 'depth': 8,
 'dropout': 0.1,
 'lags': [1, 3, 7, 14, 30, 60],
 'rolls': [7, 30],
 'n_train_rows': 2292375,
 'n_val_rows': 60271,
 'n_test_rows': 77731,
 'gpu_name': 'NVIDIA H100 80GB HBM3'}

### Eval

In [17]:
def predict_all(model, loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE, non_blocking=True)
            pred = model(Xb).cpu().numpy()
            preds.append(pred)
            trues.append(yb.numpy())
    return np.vstack(preds), np.vstack(trues)

pred, true = predict_all(model, test_loader)

metrics = {}
for i, name in enumerate(target_cols):
    yhat = pred[:, i]
    y    = true[:, i]
    metrics[name] = {
        "MAE": float(mean_absolute_error(y, yhat)),
        "RMSE": float(np.sqrt(mean_squared_error(y, yhat))),
    }
metrics

{'tmax_target': {'MAE': 2.4343862533569336, 'RMSE': 4.622142218137861},
 'tmin_target': {'MAE': 1.9923429489135742, 'RMSE': 3.7069906025498067},
 'prcp_target': {'MAE': 0.11914080381393433, 'RMSE': 0.28676113106475803}}

In [18]:
err = pred - true
df_city = pd.DataFrame({
    "city_id": test_city_ids,
    "abs_tmax": np.abs(err[:,0]),
    "abs_tmin": np.abs(err[:,1]),
    "abs_prcp": np.abs(err[:,2]),
    "sq_tmax": err[:,0]**2,
    "sq_tmin": err[:,1]**2,
    "sq_prcp": err[:,2]**2,
})
city_metrics = df_city.groupby("city_id").agg(
    MAE_tmax=("abs_tmax","mean"),
    MAE_tmin=("abs_tmin","mean"),
    MAE_prcp=("abs_prcp","mean"),
    RMSE_tmax=("sq_tmax", lambda x: float(np.sqrt(np.mean(x)))),
    RMSE_tmin=("sq_tmin", lambda x: float(np.sqrt(np.mean(x)))),
    RMSE_prcp=("sq_prcp", lambda x: float(np.sqrt(np.mean(x)))),
).reset_index()

city_metrics.sort_values("MAE_tmax").head(10)

,city_id,MAE_tmax,MAE_tmin,MAE_prcp,RMSE_tmax,RMSE_tmin,RMSE_prcp
12,USW00014923,1.194125,1.067033,0.114293,3.135793,2.643461,0.245632
1,USW00003820,1.553950,1.574216,0.168971,3.206029,3.261494,0.442461
6,USW00014710,1.746463,1.340865,0.142096,3.873437,3.024940,0.295171
0,USW00003812,1.786275,1.420904,0.160469,3.707218,2.962761,0.370235
21,USW00093805,1.853274,1.969754,0.180666,3.356741,3.708825,0.413304
23,USW00094847,1.913063,1.459644,0.130243,3.884221,2.833026,0.264196
14,USW00023179,1.916285,1.715861,0.052940,3.258923,2.665708,0.085238
22,USW00093842,1.931421,1.840958,0.181140,3.523744,3.365942,0.441064
2,USW00003860,1.949839,1.558188,0.142799,3.988521,3.179616,0.278551
20,USW00093729,1.964404,1.985261,0.256972,3.481086,3.530029,0.512864


In [19]:
os.makedirs("outputs/models", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)

torch.save(model.state_dict(), "outputs/models/mlp_state.pt")
pd.DataFrame(epoch_rows).to_csv("outputs/metrics/history.csv", index=False)
with open("outputs/metrics/test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

import joblib
joblib.dump(scaler_X, "outputs/models/feature_scaler.pkl")

['outputs/models/feature_scaler.pkl']

In [20]:
with open("results/benchmarks/run_metadata.json") as f:
    meta = json.load(f)
with open("outputs/metrics/run_summary.json") as f:
    summ = json.load(f)

row = {
    "Run ID": meta["run_id"],
    "Platform": meta["platform_label"],
    "Instance": meta["instance_label"],
    "CPU/GPU": meta["cpu_or_gpu"],
    "GPU Name": meta.get("gpu_name", ""),
    "Time per Epoch (sec)": summ["avg_epoch_sec"],
    "Total Training Time (min)": summ["total_training_min"],
    "Peak Memory (MB)": summ.get("peak_rss_mb", None),
    "GPU Utilization": "N/A (CPU)" if meta["cpu_or_gpu"] == "CPU" else "See results/benchmarks/gpu_metrics.csv",
    "Cost per Run": "",        # fill later
    "Speedup vs CPU": "",      # compute after GPU run exists
}

df = pd.DataFrame([row])
out = "results/benchmarks/benchmark_row.csv"
if os.path.exists(out):
    df.to_csv(out, mode="a", header=False, index=False)
else:
    df.to_csv(out, index=False)
df

,Run ID,Platform,Instance,CPU/GPU,GPU Name,Time per Epoch (sec),Total Training Time (min),Peak Memory (MB),GPU Utilization,Cost per Run,Speedup vs CPU
0,20260303_005030Z,Anvil,h004.anvil.rcac.purdue.edu,GPU,NVIDIA H100 80GB HBM3,4.783355,3.986367,None,See results/benchmarks/gpu_metrics.csv,,


In [21]:
from datetime import timedelta

model.eval()

demo_city = model_df["city_id"].iloc[0]
city_df = model_df[model_df["city_id"] == demo_city].sort_values("date")

last_row = city_df.iloc[-1] 
x = last_row[feature_cols].to_numpy(dtype=np.float32)
x_s = scaler_X.transform(x.reshape(1, -1))

x_t = torch.from_numpy(x_s).to(DEVICE)
with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        yhat = model(x_t).detach().cpu().numpy().ravel()

next_day = last_row["date"] + timedelta(days=1)
print(f"City {demo_city} forecast for {next_day.date()}:")
print(" tmax_pred:", float(yhat[0]))
print(" tmin_pred:", float(yhat[1]))
print(" prcp_pred:", float(yhat[2]))

City USW00003812 forecast for 2024-01-01:
 tmax_pred: 54.34375
 tmin_pred: 31.375
 prcp_pred: 0.04150390625


/tmp/ipykernel_2590500/2542014607.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
